# 第 11 章 サポートベクターマシンとカーネル法

できるだけ広い余白（マージン）を空ける境界線を選びます。カーネルで XOR も解きます。

対応する記事: [第 11 章 サポートベクターマシンとカーネル法（Python 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/python/ch11.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch05_perceptron import *
from grokking_ml.ch11_svm import *

## パーセプトロンのマージンは 0

第 5 章のパーセプトロンは、分離できた時点で更新を止めます。**その境界線がぎりぎりでも構いません。**

実際に測ると、境界線の上にちょうど乗っている点があります。正解率は 1.0 なのに、その点が少しでも動けば誤分類になります。

In [2]:
points = [(1.0, 0.0), (0.0, 2.0), (1.0, 1.0), (1.0, 2.0),
          (1.0, 3.0), (2.0, 2.0), (2.0, 3.0), (3.0, 2.0)]
labels = [-1, -1, -1, -1, 1, 1, 1, 1]
perceptron_labels = [0, 0, 0, 0, 1, 1, 1, 1]

perceptron, _ = perceptron_algorithm(points, perceptron_labels)
as_svm = SupportVectorMachine(tuple(perceptron.weights), perceptron.bias)

print(f"パーセプトロン 正解率 {accuracy(as_svm, points, labels):.2f}  "
      f"マージン {as_svm.margin(points, labels):.4f}")

パーセプトロン 正解率 1.00  マージン 0.0000


## SVM は余白を稼ぐ

**正解率は同じ 1.0 でも、マージンがまったく違います。** ヒンジ損失が「正解しているのにマージンの内側にいる点」も押し返すためです。

In [3]:
svm, errors = train_svm(points, labels, epochs=20000, regularization=0.01)

print("重み  ", [round(w, 4) for w in svm.weights])
print(f"バイアス {svm.bias:.4f}")
print(f"正解率  {accuracy(svm, points, labels):.2f}")
print(f"マージン {svm.margin(points, labels):.4f}")

重み   [1.6973, 1.6679]
バイアス -5.9300
正解率  1.00
マージン 0.6480


## ヒンジ損失は「正解しているのに損失が残る」

第 5 章のパーセプトロン誤差は正解した点を無視しました。**ヒンジ損失はマージンの外に出るまで押し続けます。**

In [4]:
sample = SupportVectorMachine(weights=(1.0, 1.0), bias=-3.0)

print(f"{'点':<12} {'スコア':>8} {'ラベル':>6} {'損失':>8}")
for point, label in [((3.0, 2.0), 1), ((1.5, 2.0), 1), ((2.0, 1.0), 1), ((1.0, 1.0), 1)]:
    print(f"{point!s:<12} {sample.score(point):>8.1f} {label:>6} "
          f"{hinge_loss(sample, point, label):>8.2f}")

点                 スコア    ラベル       損失
(3.0, 2.0)        2.0      1     0.00
(1.5, 2.0)        0.5      1     0.50
(2.0, 1.0)        0.0      1     1.00
(1.0, 1.0)       -1.0      1     2.00


## カーネルで XOR を解く

**アルゴリズムは一切変えず、カーネル関数を差し替えるだけ** で XOR が解けます。線形カーネル（ただの内積）では解けません。

In [5]:
xor_points = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]
xor_labels = [-1, 1, 1, -1]

for name, kernel in [("線形", linear_kernel),
                     ("多項式 2 次", polynomial_kernel(degree=2)),
                     ("RBF", rbf_kernel(gamma=1.0))]:
    m = train_kernel_classifier(xor_points, xor_labels, kernel=kernel)
    print(f"{name:<12} 正解率 {kernel_accuracy(m, xor_points, xor_labels):.2f}")

線形           正解率 0.50
多項式 2 次      正解率 1.00
RBF          正解率 1.00


## 試してみる: 正則化とマージン

**直感に反しますが、正則化を強くするとマージンは狭くなります。** 重みを潰しすぎると、境界線からの距離そのものが縮むためです。

In [6]:
for strength in [0.005, 0.01, 0.05, 0.1, 0.5]:
    m, _ = train_svm(points, labels, epochs=20000, regularization=strength)
    print(f"λ = {strength:<6} マージン {m.margin(points, labels):.4f}  "
          f"正解率 {accuracy(m, points, labels):.2f}")

λ = 0.005  マージン 0.6841  正解率 1.00


λ = 0.01   マージン 0.6480  正解率 1.00


λ = 0.05   マージン 0.4007  正解率 1.00


λ = 0.1    マージン 0.3848  正解率 1.00


λ = 0.5    マージン 0.1866  正解率 1.00
